# Notebook 02 — Generate the German Question Set with Ollama (full-document version)

## Purpose

This notebook turns the six German Botschaften prepared by Notebook 01 into a question set for the thesis. In this version the model reads the **complete Botschaft in one prompt** (no chunking), so questions can draw on any part of the document, including comparative questions that connect different sections.

Expected project layout:

```text
project/
├── data/
│   ├── extracted_text/          # from Notebook 01
│   └── questions/               # written by this notebook
│       └── cache/               # raw model responses, lets you resume an interrupted run
└── notebooks/
```

Run from the project root, or change `PROJECT_ROOT` below.

## 1. Configuration

All run settings live in this cell and are saved to `data/questions/run_config.json` at the end, so the exact configuration of the run can be reported in the thesis.

Notes on the choices:

* **`num_ctx` is not set here.** It is computed in Section 3 from the longest document, then used for *every* call. Changing `num_ctx` between calls would make Ollama reload the model.
* **`THINK = False`** asks Ollama to disable the reasoning phase for these extraction-style steps. If the installed `qwen3-next:80b` build is a thinking-only variant, the notebook detects the refusal, continues without the flag, and strips any `<think>…</think>` block from the response. In that case raise `MAX_OUTPUT_TOKENS_GENERATION` (reasoning tokens count against the output budget).
* **Temperatures** are low because the tasks are extraction and selection. The model's own defaults (temperature 0.6, top_k 20, top_p 0.95) are left unchanged for the other sampling parameters and are recorded in the run configuration.
* **`QUOTA`** fixes the question-type mix per Botschaft (4 factual, 3 explanatory, 3 comparative = 10, inside the planned 8–10 range).

In [14]:
from pathlib import Path
import json
import math
import re
import time
import unicodedata
import requests
import pandas as pd

# ------------------------------------------------------------
# Model and server
# ------------------------------------------------------------
MODEL_NAME = "qwen3-next:80b"          # change this variable to switch models
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_GENERATE_URL = f"{OLLAMA_BASE_URL}/api/generate"
KEEP_ALIVE = "60m"                     # keep the 50 GB model loaded between calls

# ------------------------------------------------------------
# Decoding settings (recorded in run_config.json)
# ------------------------------------------------------------
SEED = 42
TEMPERATURE_GENERATION = 0.2
TEMPERATURE_SELECTION = 0.0
TEMPERATURE_TRANSLATION = 0.0
THINK = False                          # False = no reasoning phase; None = leave to the model default
MAX_OUTPUT_TOKENS_GENERATION = 8000    # includes reasoning tokens if the model cannot disable thinking
MAX_OUTPUT_TOKENS_SELECTION = 2000
MAX_OUTPUT_TOKENS_TRANSLATION = 600

# ------------------------------------------------------------
# Context window
# ------------------------------------------------------------
NUM_CTX_OVERRIDE = None                # set an int to force a value, otherwise computed in Section 3
TOKEN_CHARS_CONSERVATIVE = 3.0         # characters per token used for sizing; German text is usually ~3.5-4
PROMPT_OVERHEAD_TOKENS = 3000          # instructions + JSON schema

# ------------------------------------------------------------
# Question design
# ------------------------------------------------------------
QUESTION_TYPES = ["factual", "explanatory", "comparative"]
CANDIDATES_PER_TYPE = 6                # candidates requested per type and document
QUOTA = {"factual": 4, "explanatory": 3, "comparative": 3}   # selected per document
MIN_QUESTIONS_PER_DOCUMENT = 8
MAX_QUESTIONS_PER_DOCUMENT = 10
ALLOW_PARTIAL_EVIDENCE = True          # keep candidates whose quotation is only partly found (e.g. spans a page break)
DUPLICATE_JACCARD_THRESHOLD = 0.8

# ------------------------------------------------------------
# Robustness and bookkeeping
# ------------------------------------------------------------
REQUEST_TIMEOUT_SECONDS = 3600         # a full-document prompt can take minutes to prefill
MAX_RETRIES = 3
USE_CACHE = True                       # reuse saved raw responses if the same call was already made
PROMPT_VERSION = "v2-fulldoc-2026-10"

PROJECT_ROOT = Path.cwd()
TEXT_DIR = PROJECT_ROOT / "../data" / "extracted_text"
QUESTION_DIR = PROJECT_ROOT / "../data" / "questions"
CACHE_DIR = QUESTION_DIR / "cache"
for d in (QUESTION_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

assert sum(QUOTA.values()) <= MAX_QUESTIONS_PER_DOCUMENT
assert sum(QUOTA.values()) >= MIN_QUESTIONS_PER_DOCUMENT
assert set(QUOTA) == set(QUESTION_TYPES)

print("Project root:", PROJECT_ROOT)
print("Model:", MODEL_NAME, "| prompt version:", PROMPT_VERSION)
print("Quota per document:", QUOTA, "=", sum(QUOTA.values()))

Project root: /home/bern/Documents/llm_trustworthiness/scripts
Model: qwen3-next:80b | prompt version: v2-fulldoc-2026-10
Quota per document: {'factual': 4, 'explanatory': 3, 'comparative': 3} = 10


## 2. Check Ollama, the model and the corpus

This cell verifies that Ollama is reachable, the model is installed, and reads the model's **digest, quantisation, capabilities and maximum context length** from Ollama itself, so that the numbers in the thesis come from the server and not from memory.

In [15]:
def ollama_get(path):
    r = requests.get(f"{OLLAMA_BASE_URL}{path}", timeout=60)
    r.raise_for_status()
    return r.json()

def ollama_post(path, payload):
    r = requests.post(f"{OLLAMA_BASE_URL}{path}", json=payload, timeout=120)
    r.raise_for_status()
    return r.json()

tags = ollama_get("/api/tags").get("models", [])
installed = {m["name"]: m for m in tags}
print("Ollama is reachable. Installed models:", len(installed))

if MODEL_NAME not in installed:
    raise RuntimeError(
        f"Model {MODEL_NAME!r} is not installed. Installed: {sorted(installed)}. "
        "Run `ollama list` in the server terminal and update MODEL_NAME."
    )

MODEL_DIGEST = installed[MODEL_NAME].get("digest", "")
show = ollama_post("/api/show", {"model": MODEL_NAME})
details = show.get("details", {})
model_info = show.get("model_info", {})
ctx_keys = [k for k in model_info if k.endswith(".context_length")]
MODEL_MAX_CTX = int(model_info[ctx_keys[0]]) if ctx_keys else 262144
try:
    OLLAMA_VERSION = ollama_get("/api/version").get("version", "unknown")
except Exception:
    OLLAMA_VERSION = "unknown"

print("Model digest      :", MODEL_DIGEST[:12])
print("Parameter size    :", details.get("parameter_size"))
print("Quantisation      :", details.get("quantization_level"))
print("Capabilities      :", show.get("capabilities"))
print("Max context length:", MODEL_MAX_CTX)
print("Model defaults    :", " | ".join(show.get("parameters", "").split("\n")))
print("Ollama version    :", OLLAMA_VERSION)

if not TEXT_DIR.exists():
    raise FileNotFoundError(f"Could not find {TEXT_DIR}. Run Notebook 01 first from the same project root.")
json_files = sorted(TEXT_DIR.glob("*.json"))
txt_files = sorted(TEXT_DIR.glob("*.txt"))
if not json_files and not txt_files:
    raise FileNotFoundError(f"No .txt or .json corpus files found in {TEXT_DIR}.")
print(f"Found {len(txt_files)} TXT files and {len(json_files)} JSON files.")

Ollama is reachable. Installed models: 19
Model digest      : b2ebb986e4e9
Parameter size    : 79.7B
Quantisation      : Q4_K_M
Capabilities      : ['completion', 'tools', 'thinking']
Max context length: 262144
Model defaults    : top_k                          20 | top_p                          0.95 | repeat_penalty                 1 | stop                           "<|im_start|>" | stop                           "<|im_end|>" | temperature                    0.6
Ollama version    : 0.24.0
Found 6 TXT files and 6 JSON files.


## 3. Load the Botschaften and size the context window

The JSON files from Notebook 01 are preferred because they keep page numbers. Every document is assembled into one text with `[PAGE n]` markers.

Then `num_ctx` is computed once from the **longest** document (conservatively estimated at 3 characters per token) plus prompt overhead and the output budget, rounded up to a multiple of 4096 and capped at the model maximum. The same value is used for all calls. If the longest document does not fit, the notebook stops instead of letting Ollama silently truncate the prompt.

In [16]:
def load_json_document(path: Path) -> dict:
    payload = json.loads(path.read_text(encoding="utf-8"))
    pages = []
    for page in payload.get("pages", []):
        text = (page.get("text") or "").strip()
        if text:
            pages.append({"page": int(page["page"]) if page.get("page") is not None else None, "text": text})
    return {
        "document_id": payload.get("selection_key") or path.stem,
        "title": payload.get("title") or path.stem,
        "date": payload.get("date"),
        "source_file": path.name,
        "pages": pages,
    }

def parse_txt_pages(path: Path) -> dict:
    text = path.read_text(encoding="utf-8", errors="replace")
    title_match = re.search(r"^TITLE:\s*(.+)$", text, flags=re.MULTILINE)
    matches = list(re.finditer(r"===== PAGE (\d+) =====", text))
    pages = []
    for i, m in enumerate(matches):
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        page_text = text[m.end():end].strip()
        if page_text:
            pages.append({"page": int(m.group(1)), "text": page_text})
    if not pages:
        pages = [{"page": None, "text": text}]
    return {
        "document_id": path.stem,
        "title": title_match.group(1).strip() if title_match else path.stem,
        "date": None,
        "source_file": path.name,
        "pages": pages,
    }

def assemble_text(pages):
    return "\n\n".join(
        (f"[PAGE {p['page']}]\n" if p["page"] is not None else "") + p["text"] for p in pages
    )

documents, used = [], set()
for jp in json_files:
    try:
        doc = load_json_document(jp)
        if doc["pages"]:
            documents.append(doc)
            used.add(jp.stem)
    except (json.JSONDecodeError, OSError) as exc:
        print(f"WARNING: could not read {jp.name}: {exc}")
for tp in txt_files:
    if tp.stem not in used:
        doc = parse_txt_pages(tp)
        if doc["pages"]:
            documents.append(doc)
if not documents:
    raise RuntimeError("No readable document text could be loaded.")

documents = sorted(documents, key=lambda d: (d["title"].casefold(), d["document_id"]))
for doc in documents:
    doc["text"] = assemble_text(doc["pages"])
    doc["est_tokens"] = int(len(doc["text"]) / TOKEN_CHARS_CONSERVATIVE)

longest = max(d["est_tokens"] for d in documents)
needed = longest + PROMPT_OVERHEAD_TOKENS + MAX_OUTPUT_TOKENS_GENERATION
if NUM_CTX_OVERRIDE:
    NUM_CTX = int(NUM_CTX_OVERRIDE)
else:
    NUM_CTX = min(MODEL_MAX_CTX, int(math.ceil(needed / 4096) * 4096))
if needed > min(NUM_CTX, MODEL_MAX_CTX):
    raise RuntimeError(
        f"The longest document needs about {needed:,} tokens (incl. overhead and output) but num_ctx is "
        f"{min(NUM_CTX, MODEL_MAX_CTX):,}. Reduce MAX_OUTPUT_TOKENS_GENERATION or fall back to chunking."
    )

overview = pd.DataFrame([{
    "document_id": d["document_id"], "title": d["title"], "pages_with_text": len(d["pages"]),
    "characters": len(d["text"]), "est_tokens_conservative": d["est_tokens"],
    "share_of_num_ctx": round(d["est_tokens"] / NUM_CTX, 2),
} for d in documents])
display(overview)
print(f"num_ctx for ALL calls: {NUM_CTX:,}  (longest document ≈ {longest:,} tokens, model maximum {MODEL_MAX_CTX:,})")

,document_id,title,pages_with_text,characters,est_tokens_conservative,share_of_num_ctx
0,13_ahv_rente,Botschaft zur Volksinitiative «Für ein bessere...,28,74703,24901,0.22
1,umweltverantwortungsinitiative,Botschaft zur Volksinitiative «Für eine verant...,20,50122,16707,0.15
2,steuergerechtigkeits_initiative,Botschaft zur Volksinitiative «Für eine zivils...,108,274951,91650,0.80
3,stopfleber_initiative,Botschaft zur Volksinitiative «Ja zum Importve...,16,44034,14678,0.13
4,nachhaltigkeitsinitiative,Botschaft zur Volksinitiative «Keine 10-Millio...,110,299015,99671,0.87
5,neutralitaetsinitiative,Botschaft zur Volksinitiative «Wahrung der sch...,30,80586,26862,0.23


num_ctx for ALL calls: 114,688  (longest document ≈ 99,671 tokens, model maximum 262,144)


## 4. Helper functions: Ollama calls, JSON parsing, evidence verification

* `call_ollama` sends every request with the same `num_ctx`, seed and keep-alive, handles models that refuse the `think` flag, strips `<think>` blocks, retries on malformed JSON, and logs token counts and durations to `CALL_LOG`.
* Raw responses are cached on disk. If a long run is interrupted, re-running the notebook reuses finished calls (switch off with `USE_CACHE = False`, or change `PROMPT_VERSION`, to force fresh calls).
* Evidence verification normalises both the quotation and the page text to lowercase letters and digits only, so line-break hyphenation, spacing, punctuation and quotation-mark style do not cause false mismatches.

In [17]:
CALL_LOG = []
_THINK_SUPPORTED = {"flag": THINK is not None}

def strip_think(text: str) -> str:
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

def extract_json_object(text: str):
    text = strip_think(text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if m:
            return json.loads(m.group(0))
        raise

def safe_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", value)[:150]

def call_ollama(prompt, *, temperature, num_predict, label):
    est_prompt_tokens = len(prompt) / TOKEN_CHARS_CONSERVATIVE
    if est_prompt_tokens + num_predict > NUM_CTX:
        raise RuntimeError(
            f"[{label}] prompt ≈{int(est_prompt_tokens):,} tokens + output {num_predict:,} exceeds num_ctx {NUM_CTX:,}."
        )

    cache_file = CACHE_DIR / f"{safe_name(MODEL_NAME)}__{PROMPT_VERSION}__{safe_name(label)}.json"
    if USE_CACHE and cache_file.exists():
        return json.loads(cache_file.read_text(encoding="utf-8"))

    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "format": "json",
        "keep_alive": KEEP_ALIVE,
        "options": {
            "temperature": temperature,
            "num_ctx": NUM_CTX,
            "num_predict": num_predict,
            "seed": SEED,
        },
    }
    last_error = None
    attempt = 0
    while attempt < MAX_RETRIES:
        body = dict(payload)
        if _THINK_SUPPORTED["flag"] and THINK is not None:
            body["think"] = THINK
        try:
            started = time.time()
            r = requests.post(OLLAMA_GENERATE_URL, json=body, timeout=(30, REQUEST_TIMEOUT_SECONDS))
            if r.status_code == 400 and "think" in r.text.lower() and "think" in body:
                print(f"[{label}] this model build does not accept the think flag; continuing without it.")
                _THINK_SUPPORTED["flag"] = False
                continue                      # retry immediately without counting an attempt
            r.raise_for_status()
            result = r.json()
            CALL_LOG.append({
                "label": label,
                "prompt_eval_count": result.get("prompt_eval_count"),
                "eval_count": result.get("eval_count"),
                "done_reason": result.get("done_reason"),
                "total_seconds": round(result.get("total_duration", 0) / 1e9, 1),
                "wall_seconds": round(time.time() - started, 1),
            })
            if result.get("done_reason") == "length":
                raise RuntimeError("output hit the token limit (raise the MAX_OUTPUT_TOKENS_* value)")
            raw = (result.get("response") or "")
            if not strip_think(raw):
                raise RuntimeError("empty response (the model may have used the whole budget for reasoning)")
            parsed = extract_json_object(raw)
            cache_file.write_text(json.dumps(parsed, ensure_ascii=False, indent=2), encoding="utf-8")
            return parsed
        except Exception as exc:
            attempt += 1
            last_error = exc
            print(f"[{label}] request failed (attempt {attempt}/{MAX_RETRIES}): {exc}")
            if attempt < MAX_RETRIES:
                time.sleep(3 * attempt)
    raise RuntimeError(f"[{label}] failed after {MAX_RETRIES} attempts: {last_error}")


# ---------- evidence verification ----------
def alnum_norm(text: str) -> str:
    text = unicodedata.normalize("NFKC", str(text)).casefold()
    return re.sub(r"[\W_]+", "", text)

def build_page_index(doc):
    return {p["page"]: alnum_norm(p["text"]) for p in doc["pages"] if p["page"] is not None}

def verify_quote(quote, cited_page, page_index):
    # Returns (status, pages_found); status: exact | exact_other_page | partial | not_found
    q = alnum_norm(quote)
    if len(q) < 25:
        return "not_found", []
    found = [pg for pg, txt in page_index.items() if q in txt]
    if found:
        try:
            cited = int(cited_page)
        except (TypeError, ValueError):
            cited = None
        return ("exact" if cited in found else "exact_other_page"), found
    # partial: start and end of the quotation both occur (e.g. quotation spans a page break)
    head, tail = q[:40], q[-40:]
    pages_head = [pg for pg, txt in page_index.items() if head in txt]
    pages_tail = [pg for pg, txt in page_index.items() if tail in txt]
    if pages_head and pages_tail:
        return "partial", sorted(set(pages_head + pages_tail))
    return "not_found", []

def word_set(text):
    return set(re.findall(r"\w+", text.casefold()))

def jaccard(a, b):
    return len(a & b) / len(a | b) if a and b else 0.0

## 5. Generate German candidate questions from the complete Botschaft

For each Botschaft there is **one call per question type** (factual, explanatory, comparative). The Botschaft comes first in the prompt and the instructions last, so the three calls share an identical prefix and Ollama can reuse the already-processed document instead of reading it three times.

The model does **not** write a reference answer. For every candidate it returns only:

* the question (German),
* one to three **verbatim evidence quotations** with their page numbers.

Comparative questions may cite quotations from different pages, which is the main benefit of giving the model the whole document.

In [18]:
TYPE_GUIDE = {
    "factual": (
        "factual recall: a specific fact, actor, date, figure, measure or decision that is stated in the Botschaft "
        "(who/what/when/how many/which measure)."
    ),
    "explanatory": (
        "explanatory: the reasons, aims, mechanisms or consequences that the Botschaft itself gives for something "
        "(why/how/for what purpose/with what effect), attributed to whoever states them."
    ),
    "comparative": (
        "comparative or relational: a comparison or relationship that the Botschaft explicitly draws or clearly supports, "
        "for example the position of the initiative committee versus the position of the Federal Council, the current "
        "situation versus the proposed change, or two measures/options against each other. Use evidence from at least "
        "two different places in the document where possible."
    ),
}

def document_block(doc):
    return (
        f"DOKUMENTTITEL: {doc['title']}\n"
        f"DATUM: {doc.get('date') or 'unbekannt'}\n\n"
        f"<BOTSCHAFT>\n{doc['text']}\n</BOTSCHAFT>"
    )

def candidate_instructions(question_type, n):
    return f'''You are helping to construct a research question set for evaluating answers that are grounded in official Swiss Federal Council Botschaften. The complete Botschaft is given above. Page numbers are marked as [PAGE n].

Use ONLY the Botschaft above. Do not use outside knowledge and do not invent facts.

Write {n} distinct candidate questions of this type: {TYPE_GUIDE[question_type]}

Requirements:
- Write every question in German.
- Each question must be clear, neutral and answerable from the Botschaft alone.
- Each question must be understandable without the document at hand: name the initiative or measure explicitly and never write phrases such as "im Text", "in diesem Abschnitt" or "gemäss der Botschaft".
- Spread the questions over different parts of the document; do not cluster them on one page or topic.
- Avoid yes/no questions, vague questions, near-duplicates and questions that need outside knowledge.
- Do not present political positions as facts. If a position belongs to the Federal Council, the initiative committee, Parliament or another actor, name that actor in the question.
- Do NOT write answers or reference answers.
- For each question give 1 to 3 evidence quotations. Each quotation must be copied verbatim from a single page of the Botschaft (8 to 40 words, exactly as printed, without ellipses) and must contain the information needed to answer the question. Give the page number from the nearest [PAGE n] marker above the quotation.

Return valid JSON only, in exactly this structure:
{{"questions":[{{"question_de":"...","evidence":[{{"quote_de":"...","page":12}}]}}]}}'''

candidate_rows = []
for d_idx, doc in enumerate(documents, start=1):
    page_index = build_page_index(doc)
    prefix = document_block(doc)
    print(f"[{d_idx}/{len(documents)}] {doc['title']}  (≈{doc['est_tokens']:,} tokens)")
    for qtype in QUESTION_TYPES:
        prompt = prefix + "\n\n" + candidate_instructions(qtype, CANDIDATES_PER_TYPE)
        result = call_ollama(
            prompt, temperature=TEMPERATURE_GENERATION, num_predict=MAX_OUTPUT_TOKENS_GENERATION,
            label=f"{doc['document_id']}__candidates__{qtype}",
        )
        items = result.get("questions", [])
        if not isinstance(items, list):
            print(f"   WARNING: unexpected output for {qtype}")
            continue
        n_ok = 0
        for item in items:
            q = str(item.get("question_de", "")).strip()
            evidence = item.get("evidence", [])
            if not q or not isinstance(evidence, list) or not evidence:
                continue
            checks, quotes, pages = [], [], []
            for ev in evidence[:3]:
                if not isinstance(ev, dict):
                    continue
                quote = str(ev.get("quote_de", "")).strip()
                status, found_pages = verify_quote(quote, ev.get("page"), page_index)
                checks.append(status)
                quotes.append(quote)
                pages.extend(found_pages if found_pages else ([int(ev["page"])] if str(ev.get("page", "")).isdigit() else []))
            if all(c in ("exact", "exact_other_page") for c in checks):
                evidence_status = "verified"
            elif all(c in ("exact", "exact_other_page", "partial") for c in checks):
                evidence_status = "partial"
            else:
                evidence_status = "failed"
            n_ok += evidence_status == "verified"
            candidate_rows.append({
                "document_id": doc["document_id"],
                "document_title": doc["title"],
                "source_file": doc["source_file"],
                "question_type": qtype,
                "question_de": q,
                "evidence_quotes_de": json.dumps(quotes, ensure_ascii=False),
                "source_pages": json.dumps(sorted(set(pages))),
                "evidence_status": evidence_status,
                "evidence_checks": json.dumps(checks),
            })
        print(f"   {qtype:12s}: {len(items)} proposed, {n_ok} with verified evidence")

candidates = pd.DataFrame(candidate_rows)
if candidates.empty:
    raise RuntimeError("No candidate questions were generated. Inspect the cache folder and CALL_LOG.")

# duplicate removal within a document (exact normalised text and word-overlap)
keep = []
for doc_id, grp in candidates.groupby("document_id", sort=False):
    kept_sets = []
    for idx, row in grp.iterrows():
        ws = word_set(row["question_de"])
        if any(jaccard(ws, other) >= DUPLICATE_JACCARD_THRESHOLD for other in kept_sets):
            continue
        kept_sets.append(ws)
        keep.append(idx)
candidates = candidates.loc[keep].reset_index(drop=True)
candidates.insert(0, "candidate_id", range(len(candidates)))

candidate_path = QUESTION_DIR / "question_candidates_german.csv"
candidates.to_csv(candidate_path, index=False, encoding="utf-8-sig")
print(f"\nSaved {len(candidates)} unique candidates (all evidence statuses) to: {candidate_path}")
display(candidates.groupby(["document_title", "question_type", "evidence_status"]).size().rename("n").reset_index())

[1/6] Botschaft zur Volksinitiative «Für ein besseres Leben im Alter (Initiative für eine 13. AHV-Rente)»  (≈24,901 tokens)
   factual     : 6 proposed, 6 with verified evidence
   explanatory : 6 proposed, 5 with verified evidence
   comparative : 6 proposed, 6 with verified evidence
[2/6] Botschaft zur Volksinitiative «Für eine verantwortungsvolle Wirtschaft innerhalb der planetaren Grenzen (Umweltverantwortungsinitiative)»  (≈16,707 tokens)
   factual     : 6 proposed, 5 with verified evidence
   explanatory : 6 proposed, 4 with verified evidence
   comparative : 6 proposed, 2 with verified evidence
[3/6] Botschaft zur Volksinitiative «Für eine zivilstandsunabhängige Individualbesteuerung (Steuergerechtigkeits-Initiative)» und zum indirekten Gegenvorschlag (Bundesgesetz über die Individualbesteuerung)  (≈91,650 tokens)
   factual     : 6 proposed, 6 with verified evidence
   explanatory : 6 proposed, 5 with verified evidence
   comparative : 6 proposed, 6 with verified evidence
[4/6

,document_title,question_type,evidence_status,n
0,Botschaft zur Volksinitiative «Für ein bessere...,comparative,verified,6
1,Botschaft zur Volksinitiative «Für ein bessere...,explanatory,partial,1
2,Botschaft zur Volksinitiative «Für ein bessere...,explanatory,verified,5
3,Botschaft zur Volksinitiative «Für ein bessere...,factual,verified,6
4,Botschaft zur Volksinitiative «Für eine verant...,comparative,failed,1
5,Botschaft zur Volksinitiative «Für eine verant...,comparative,partial,2
6,Botschaft zur Volksinitiative «Für eine verant...,comparative,verified,2
7,Botschaft zur Volksinitiative «Für eine verant...,explanatory,failed,2
8,Botschaft zur Volksinitiative «Für eine verant...,explanatory,verified,3
9,Botschaft zur Volksinitiative «Für eine verant...,factual,partial,1


## 6. Select the question set per Botschaft

Only candidates whose quotation was found in the source are eligible (`verified`, plus `partial` if `ALLOW_PARTIAL_EVIDENCE` is on). The model picks the best candidates **per question type** and the number per type is fixed by `QUOTA`, so the type mix is enforced in code instead of only requested in a prompt.

The selection prompt does not contain the document: it only sees the candidate questions and their page numbers, and it may not rewrite anything. Invalid or missing IDs fall back to the first eligible candidates of that type. Shortfalls (fewer eligible candidates than the quota) are reported, never filled with invented questions.

In [19]:
def selection_prompt(doc_title, qtype, subset, k):
    records = [{
        "candidate_id": int(r["candidate_id"]),
        "question_de": r["question_de"],
        "source_pages": json.loads(r["source_pages"]),
    } for _, r in subset.iterrows()]
    return f'''Select {k} questions for a research benchmark from the candidates below. They all come from one official Swiss Botschaft and are all of the type "{qtype}".

Select for:
- clear answerability from the cited pages;
- coverage of distinct, important points in different parts of the document (prefer different source pages);
- non-redundancy and neutral wording; attribution of positions to the right actor.

Do not rewrite questions and do not create new ones. Return only candidate IDs, without duplicates. If fewer than {k} candidates are suitable, return the suitable ones.

Return valid JSON only: {{"selected_candidate_ids":[0,1,2]}}

Document: {doc_title}

CANDIDATES:
{json.dumps(records, ensure_ascii=False, indent=2)}
'''

eligible_status = {"verified", "partial"} if ALLOW_PARTIAL_EVIDENCE else {"verified"}
selected_frames, shortfalls = [], []

for doc in documents:
    for qtype in QUESTION_TYPES:
        k = QUOTA[qtype]
        pool = candidates[
            (candidates["document_id"] == doc["document_id"])
            & (candidates["question_type"] == qtype)
            & (candidates["evidence_status"].isin(eligible_status))
        ]
        if pool.empty:
            shortfalls.append((doc["title"], qtype, k, 0))
            continue
        if len(pool) <= k:
            chosen = pool.copy()
        else:
            result = call_ollama(
                selection_prompt(doc["title"], qtype, pool, k),
                temperature=TEMPERATURE_SELECTION, num_predict=MAX_OUTPUT_TOKENS_SELECTION,
                label=f"{doc['document_id']}__selection__{qtype}",
            )
            valid_ids, allowed = [], set(pool["candidate_id"])
            for value in result.get("selected_candidate_ids", []):
                try:
                    value = int(value)
                except (TypeError, ValueError):
                    continue
                if value in allowed and value not in valid_ids:
                    valid_ids.append(value)
            for fallback in pool["candidate_id"]:          # top up with the first eligible candidates
                if len(valid_ids) >= k:
                    break
                if fallback not in valid_ids:
                    valid_ids.append(int(fallback))
            chosen = pool[pool["candidate_id"].isin(valid_ids[:k])].copy()
        if len(chosen) < k:
            shortfalls.append((doc["title"], qtype, k, len(chosen)))
        selected_frames.append(chosen)

if not selected_frames:
    raise RuntimeError("The selection step returned no questions.")

selected_questions = pd.concat(selected_frames, ignore_index=True)
selected_questions["selection_method"] = f"{MODEL_NAME} selection per type from evidence-verified candidates"
selected_questions = selected_questions.sort_values(["document_id", "question_type", "candidate_id"]).reset_index(drop=True)
selected_questions["question_id"] = (
    selected_questions["document_id"].astype(str) + "_Q"
    + (selected_questions.groupby("document_id").cumcount() + 1).astype(str).str.zfill(2)
)

print("Selected questions:", len(selected_questions))
if shortfalls:
    print("\nSHORTFALLS (fewer eligible candidates than the quota):")
    for t in shortfalls:
        print(f"  {t[0]} | {t[1]}: wanted {t[2]}, got {t[3]}")
display(selected_questions.groupby(["document_title", "question_type"]).size().rename("count").reset_index())

Selected questions: 60


,document_title,question_type,count
0,Botschaft zur Volksinitiative «Für ein bessere...,comparative,3
1,Botschaft zur Volksinitiative «Für ein bessere...,explanatory,3
2,Botschaft zur Volksinitiative «Für ein bessere...,factual,4
3,Botschaft zur Volksinitiative «Für eine verant...,comparative,3
4,Botschaft zur Volksinitiative «Für eine verant...,explanatory,3
5,Botschaft zur Volksinitiative «Für eine verant...,factual,4
6,Botschaft zur Volksinitiative «Für eine zivils...,comparative,3
7,Botschaft zur Volksinitiative «Für eine zivils...,explanatory,3
8,Botschaft zur Volksinitiative «Für eine zivils...,factual,4
9,Botschaft zur Volksinitiative «Ja zum Importve...,comparative,3


## 7. Translate the selected questions into English

The German question stays the authoritative question for answer generation and evaluation. The English translation is a convenience column for documentation and for your own review, produced by the same model without access to the document.

In [20]:
def translate_prompt(question_de):
    return f'''Translate the following question from German into natural, faithful English.

Rules:
- Preserve the meaning, scope, names, dates, quantities and attribution exactly.
- Do not answer the question.
- Do not add context or explanatory text.
- Return valid JSON only in this format:
{{"question_en":"..."}}

German question:
{question_de}
'''

translations = []
for idx, row in selected_questions.iterrows():
    try:
        result = call_ollama(
            translate_prompt(row["question_de"]),
            temperature=TEMPERATURE_TRANSLATION, num_predict=MAX_OUTPUT_TOKENS_TRANSLATION,
            label=f"{row['question_id']}__translation",
        )
        translation = str(result.get("question_en", "")).strip()
        if not translation:
            print(f"WARNING: empty translation for {row['question_id']}")
    except Exception as exc:
        translation = ""
        print(f"WARNING: translation failed for {row['question_id']}: {exc}")
    translations.append(translation)

selected_questions["question_en"] = translations
selected_questions["model_name"] = MODEL_NAME
selected_questions["model_digest"] = MODEL_DIGEST[:12]
selected_questions["prompt_version"] = PROMPT_VERSION
selected_questions["generated_at_utc"] = pd.Timestamp.now(tz="UTC").isoformat()

## 8. Save the outputs

Files written to `data/questions/`:

| File | Content |
|---|---|
| `question_set_german_english.csv` | final candidate set for manual review (German + English, type, evidence quotations, verified pages) |
| `question_set_review.xlsx` | same data with empty review columns, plus a per-document summary |
| `selected_questions_before_translation.csv` | selection output with all verification fields |
| `question_candidates_german.csv` | the full candidate pool incl. failed-evidence candidates, for traceability |
| `run_config.json` | model, digest, quantisation, `num_ctx`, seed, temperatures, quota, prompt version, versions |
| `ollama_call_log.csv` | prompt/output token counts and durations of every call (useful for the computing-environment section) |


In [21]:
preferred_columns = [
    "question_id", "document_id", "document_title", "question_type",
    "question_de", "question_en",
    "evidence_quotes_de", "source_pages", "evidence_status",
    "source_file", "candidate_id", "model_name", "model_digest", "prompt_version",
    "generated_at_utc", "selection_method",
]
final_questions = selected_questions[[c for c in preferred_columns if c in selected_questions.columns]].copy()

final_path = QUESTION_DIR / "question_set_german_english.csv"
selected_path = QUESTION_DIR / "selected_questions_before_translation.csv"
review_path = QUESTION_DIR / "question_set_review.xlsx"
config_path = QUESTION_DIR / "run_config.json"
calllog_path = QUESTION_DIR / "ollama_call_log.csv"

final_questions.to_csv(final_path, index=False, encoding="utf-8-sig")
selected_questions.to_csv(selected_path, index=False, encoding="utf-8-sig")
pd.DataFrame(CALL_LOG).to_csv(calllog_path, index=False)

run_config = {
    "model_name": MODEL_NAME, "model_digest": MODEL_DIGEST, "ollama_version": OLLAMA_VERSION,
    "parameter_size": details.get("parameter_size"), "quantization_level": details.get("quantization_level"),
    "capabilities": show.get("capabilities"), "model_max_context": MODEL_MAX_CTX,
    "model_default_parameters": show.get("parameters"),
    "num_ctx_used": NUM_CTX, "seed": SEED, "think": THINK, "think_flag_accepted": _THINK_SUPPORTED["flag"],
    "temperature_generation": TEMPERATURE_GENERATION, "temperature_selection": TEMPERATURE_SELECTION,
    "temperature_translation": TEMPERATURE_TRANSLATION,
    "max_output_tokens": {"generation": MAX_OUTPUT_TOKENS_GENERATION, "selection": MAX_OUTPUT_TOKENS_SELECTION,
                          "translation": MAX_OUTPUT_TOKENS_TRANSLATION},
    "candidates_per_type": CANDIDATES_PER_TYPE, "quota": QUOTA, "allow_partial_evidence": ALLOW_PARTIAL_EVIDENCE,
    "duplicate_jaccard_threshold": DUPLICATE_JACCARD_THRESHOLD, "prompt_version": PROMPT_VERSION,
    "input_mode": "full document per prompt, [PAGE n] markers",
    "documents": [{"document_id": d["document_id"], "title": d["title"], "pages": len(d["pages"]),
                   "characters": len(d["text"])} for d in documents],
    "run_finished_utc": pd.Timestamp.now(tz="UTC").isoformat(),
}
config_path.write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved:", final_path)
print("Saved:", config_path)
print("Saved:", calllog_path)
print("Final question count:", len(final_questions))

openpyxl is not installed; skipped the XLSX workbook. CSV outputs were saved.
Saved: /home/bern/Documents/llm_trustworthiness/scripts/../data/questions/question_set_german_english.csv
Saved: /home/bern/Documents/llm_trustworthiness/scripts/../data/questions/run_config.json
Saved: /home/bern/Documents/llm_trustworthiness/scripts/../data/questions/ollama_call_log.csv
Final question count: 60


In [22]:
display(final_questions[["question_id", "question_type", "question_de", "question_en", "source_pages", "evidence_status"]].head(20))

,question_id,question_type,question_de,question_en,source_pages,evidence_status
0,13_ahv_rente_Q01,comparative,Wie bewertet der Bundesrat die Auswirkungen ei...,How does the Federal Council assess the impact...,[10],verified
1,13_ahv_rente_Q02,comparative,Welche unterschiedlichen Auswirkungen hätte di...,What different impacts would the popular initi...,"[9, 11]",verified
2,13_ahv_rente_Q03,comparative,Wie unterscheiden sich die Finanzierungsvorsch...,How do the financing proposals of the initiati...,"[6, 8]",verified
3,13_ahv_rente_Q04,explanatory,Welche Auswirkungen hätte die Annahme der Volk...,What impact would the acceptance of the popula...,[10],verified
4,13_ahv_rente_Q05,explanatory,Wie begründet das Initiativkomitee die Notwend...,How does the initiative committee justify the ...,"[4, 5]",partial
5,13_ahv_rente_Q06,explanatory,Wie sieht die Botschaft die Auswirkungen der I...,How does the initiative impact intergeneration...,[18],verified
6,13_ahv_rente_Q07,factual,Welchen Betrag an zusätzlichen Ausgaben für di...,What amount of additional expenses for the AHV...,[10],verified
7,13_ahv_rente_Q08,factual,Welche Finanzierungsquelle schlagen die Initia...,Which funding source do the initiators propose...,[8],verified
8,13_ahv_rente_Q09,factual,Wie viele gültige Unterschriften wurden für di...,How many valid signatures were collected for t...,[3],verified
9,13_ahv_rente_Q10,factual,Wie viele Prozent der AHV-Rentenbezügerinnen u...,What percentage of AHV pension recipients (wit...,[16],verified


## 9. Quick diagnostics

Check these before the manual review: that no call was cut off, how much of the context window the longest prompt actually used, and how many selected questions rest on only partially matched evidence.

In [23]:
log = pd.DataFrame(CALL_LOG)
if log.empty:
    print("No fresh calls were made in this run (everything came from the cache).")
else:
    print("Calls this run:", len(log))
    print("Calls that hit the output limit:", int((log["done_reason"] == "length").sum()))
    print("Largest prompt_eval_count:", int(log["prompt_eval_count"].max()), "of num_ctx", NUM_CTX)
    print("Slowest call (s):", log["wall_seconds"].max())

print("\nSelected questions by evidence status:")
print(selected_questions["evidence_status"].value_counts().to_string())

spread = (selected_questions.assign(n_pages=selected_questions["source_pages"].apply(lambda s: len(json.loads(s))))
          .groupby("document_title")["source_pages"]
          .apply(lambda s: len({p for v in s for p in json.loads(v)})).rename("distinct_pages_cited"))
print("\nDistinct pages cited per document (coverage check):")
print(spread.to_string())

Calls this run: 95
Calls that hit the output limit: 0
Largest prompt_eval_count: 99533 of num_ctx 114688
Slowest call (s): 53.2

Selected questions by evidence status:
evidence_status
verified    53
partial      7

Distinct pages cited per document (coverage check):
document_title
Botschaft zur Volksinitiative «Für ein besseres Leben im Alter (Initiative für eine 13. AHV-Rente)»                                                                                                        10
Botschaft zur Volksinitiative «Für eine verantwortungsvolle Wirtschaft innerhalb der planetaren Grenzen (Umweltverantwortungsinitiative)»                                                                  12
Botschaft zur Volksinitiative «Für eine zivilstandsunabhängige Individualbesteuerung (Steuergerechtigkeits-Initiative)» und zum indirekten Gegenvorschlag (Bundesgesetz über die Individualbesteuerung)     7
Botschaft zur Volksinitiative «Ja zum Importverbot für Stopfleber (Stopfleber-Initiative)»          

In [26]:
review_sheet = final_questions.copy()
for col in ["review_answerable", "review_evidence_supports", "review_neutral", "review_translation_ok", "review_keep", "review_notes"]:
    review_sheet[col] = ""
with pd.ExcelWriter(review_path, engine="openpyxl") as writer:
    review_sheet.to_excel(writer, sheet_name="Questions", index=False)
    (final_questions.groupby(["document_title", "question_type"]).size().rename("count").reset_index()).to_excel(writer, sheet_name="Summary", index=False)
    print("Saved review workbook:", review_path)

Saved review workbook: /home/bern/Documents/llm_trustworthiness/scripts/../data/questions/question_set_review.xlsx


## 10. Mandatory manual quality review

Before the question set is used in the answer-generation notebook, review each row in `question_set_review.xlsx` and confirm:

* **Answerability:** the Botschaft contains enough information to answer the question.
* **Evidence:** the quotation on the cited page really supports an answer (the quotation text itself was already matched to the page automatically).
* **Neutrality:** no partisan premise; positions are attributed to the right actor.
* **Coverage:** factual, explanatory and comparative questions are all present and spread across the document.
* **Non-duplication:** no near-duplicates within or across documents.
* **Translation fidelity:** the English question means the same as the German original.

Remove unsuitable questions instead of keeping them to reach a target count, then freeze the file (copy it to a new name such as `question_set_frozen.csv`) and record the date.

## 11. Reproducibility notes

* The model, its digest, `num_ctx`, seed, temperatures, quota and prompt version are in `run_config.json`; every call's token counts are in `ollama_call_log.csv`.
* The German Botschaft is the only reference. No expected answers are generated or stored.
* German is the source language for generation and later answering; the English column is documentation only.
* If the prompts, the model or the source text change, change `PROMPT_VERSION`, regenerate and re-review.
* This notebook does not generate answers or evaluate answers; those are later stages.


What changed compared with the first version:

| Topic | First version | This version |
|---|---|---|
| Model | `qwen3:14b` | `qwen3-next:80b` (same model later used as judge) |
| Input | 12,000-character chunks | complete Botschaft with `[PAGE n]` markers |
| Context | Ollama default (not set) | `num_ctx` set explicitly, sized from the longest document, identical for every call |
| Evidence | quotation requested, not checked | quotation **verified automatically** against the page text; page numbers corrected from the source |
| Question types | requested in the prompt, not enforced | one generation call per type and a **quota per type** in the selection step |
| Reproducibility | none | fixed seed, temperatures, model digest, `num_ctx`, prompt version and call statistics saved to disk |
| Thinking mode | not handled | `think` switch plus stripping of `<think>` blocks before JSON parsing |

**Important:** model-generated questions are candidates, not ground truth. Review them manually before freezing the evaluation set (Section 10).